# Fine-tuning QLoRA — Professions de foi (Archelec)

**Modèle :** `meta-llama/Llama-3.2-1B-Instruct`  
**GPU cible :** Tesla T4 (fp16, pas de bf16)  
**Objectif :** apprendre au modèle à générer des professions de foi crédibles à partir d'un prompt structuré.

**Pipeline :**
1. Authentification HuggingFace
2. Chargement dataset + métadonnées
3. Construction des prompts
4. Nettoyage OCR
5. Formatage chat template
6. Chargement modèle (QLoRA 4-bit)
7. Entraînement
8. Inférence & comparaison

In [1]:
import torch

print("Version de PyTorch :", torch.__version__)
print("Version de CUDA associée :", torch.version.cuda)
print("Le GPU est-il détecté ? :", torch.cuda.is_available())

is_a2 = False
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print("Nom du GPU :", gpu_name)
    # L'instance A2 utilise généralement des GPU NVIDIA A100 ou A10
    if "A100" in gpu_name or "A10" in gpu_name or "A2" in gpu_name:
        is_a2 = True

Version de PyTorch : 2.11.0+cpu
Version de CUDA associée : None
Le GPU est-il détecté ? : False


## 1. Authentification HuggingFace

In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

# override=True force la lecture de la nouvelle valeur dans le fichier .env
load_dotenv(override=True)
login(token=os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 2. Chargement du dataset et des métadonnées

On charge les fichiers texte (professions de foi OCRisées) et le CSV de métadonnées,
puis on les joint par l'identifiant de fichier (`id`).

In [3]:
from pathlib import Path
from datasets import load_dataset
import pandas as pd

# --- Fichiers texte à charger ---
PATTERNS = [
    "data/1981/legislatives/*PF*.txt",
    "data/1988/legislatives/*PF*.txt",
    "data/1993/legislatives/*PF*.txt",
]

# On résout les chemins dans le même ordre que load_dataset
# Note : load_dataset trie les fichiers par glob, on fait pareil avec sorted()
files = []
for p in PATTERNS:
    files.extend(sorted(Path().glob(p)))

dataset = load_dataset("text", data_files=PATTERNS, split="train", sample_by="document")
print(f"Dataset chargé : {len(dataset)} documents, {len(files)} fichiers résolus")

# --- Métadonnées ---
metadata = pd.read_csv("data/archelect_search.csv")
# Colonnes utiles uniquement
COLS_META = ["id", "titulaire-prenom", "titulaire-nom", "titulaire-profession",
             "titulaire-soutien", "contexte-tour", "date", "departement-nom"]
metadata_dict = metadata[COLS_META].set_index("id").to_dict("index")

print(f"Métadonnées : {len(metadata)} entrées")

Resolving data files:   0%|          | 0/12498 [00:00<?, ?it/s]

Dataset chargé : 12498 documents, 12498 fichiers résolus
Métadonnées : 12498 entrées


In [4]:
def add_metadata(example, idx):
    """Ajoute l'id (dérivé du nom de fichier) et les métadonnées associées."""
    path = files[idx]
    doc_id = path.stem
    example["id"] = doc_id
    example["annee"] = path.parts[-3]

    # Récupération des métadonnées (avec valeur par défaut si absent)
    meta = metadata_dict.get(doc_id, {})
    example["prenom"]     = meta.get("titulaire-prenom", "non mentionné")
    example["nom"]        = meta.get("titulaire-nom", "non mentionné")
    example["profession"] = meta.get("titulaire-profession", "non mentionné")
    example["soutien"]    = meta.get("titulaire-soutien", "non mentionné")
    example["tour"]       = meta.get("contexte-tour", "non mentionné")
    example["date"]       = meta.get("date", "non mentionné")
    example["departement"]= meta.get("departement-nom", "non mentionné")
    return example

dataset = dataset.map(add_metadata, with_indices=True)
print(dataset)

Dataset({
    features: ['text', 'id', 'annee', 'prenom', 'nom', 'profession', 'soutien', 'tour', 'date', 'departement'],
    num_rows: 12498
})


## 4. Nettoyage OCR

Les textes sont issus d'OCR et contiennent des artefacts courants :
mots coupés par un tiret en fin de ligne, filigranes CEVIPOF, mentions légales, etc.

**Attention :** on ne colle PAS les tirets intra-ligne (ex. `Bourg-en-Bresse`)
car cela détruirait les noms composés. On traite uniquement les coupures
**de fin de ligne** (tiret suivi d'un saut de ligne).

In [6]:
import re

# Précompilation pour performances (appliqué sur 12k documents)
_RE_CUT_EOL    = re.compile(r'([A-Za-zÀ-ÿ]+)-\s*\n\s*([A-Za-zÀ-ÿ]+)')  # coupure fin de ligne uniquement
_RE_WATERMARK  = re.compile(r'Sciences Po / fonds CEVIPOF|[☐☒@¥]')
_RE_VU_CAND    = re.compile(r'vu\s*[,:\-]?\s*(le|la|les)\s+candidat[e]?[s]?\s*[:.]?', re.IGNORECASE)
_RE_DROP_LINE  = re.compile(
    r'^.*('
    r'imp\.?\s|imprimerie|imprimeurs|'
    r'r\.?c\.?\s|'
    r'\b\d{5}\b|'
    r'\b\d{1,2}([\s.\-]?\d{2}){3}\b'
    r').*$',
    re.IGNORECASE | re.MULTILINE
)
_RE_MULTILINE  = re.compile(r'\n{3,}')
_RE_MULTSPACE  = re.compile(r' {2,}')
_RE_ENUM       = re.compile(r'[:\n]\s*[•*>.o·]\s*', re.MULTILINE)

def clean_ocr(example):
    text = example["text"]
    text = _RE_CUT_EOL.sub(r'\1\2', text)     # recolle mots coupés en fin de ligne
    text = _RE_WATERMARK.sub("", text)          # filigranes
    text = _RE_VU_CAND.sub("", text)            # mention légale
    text = _RE_DROP_LINE.sub("", text)          # lignes techniques
    text = _RE_MULTILINE.sub('\n\n', text)      # sauts de ligne excessifs
    text = _RE_MULTSPACE.sub(' ', text)         # espaces multiples
    text = _RE_ENUM.sub("- ", text)             # normalisation listes
    example["text"] = text.strip()
    return example

dataset = dataset.map(clean_ocr, num_proc=4)
print("Nettoyage OCR terminé.")

Nettoyage OCR terminé.


In [ ]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

def normalize_party(soutien):
    # Exclusion directe des valeurs nulles ou non mentionnées
    if pd.isna(soutien) or soutien == "non mentionné":
        return "A_EXCLURE"

    # On isole le parti principal (le premier avant un éventuel point-virgule)
    # et on harmonise en minuscules pour la recherche
    s = str(soutien).lower().split(';')[0].strip()

    # 1. Extrême Droite
    if any(x in s for x in ["front national", "fn", "extrême droite", "trop d'immigrés", "nationaliste", "royaliste", "action française", "forces nouvelles"]):
        return "Extreme_Droite"

    # 2. Communiste et Extrême Gauche
    if any(x in s for x in ["communiste", "pcf", "lutte ouvrière", "lcr", "marxiste", "trotskyste", "parti des travailleurs", "rouge et vert", "psu", "parti socialiste unifié", "combat ouvrier", "alternative démocratie socialisme"]):
        return "Communiste_et_Extreme_Gauche"

    # 3. Écologiste (On exclut spécifiquement CPNT de cette catégorie)
    if any(x in s for x in ["écolog", "ecolog", "vert", "environnement", "amis de la terre", "nature et animaux", "biosphère"]) and "chasse" not in s:
        return "Ecologiste"

    # 4. Socialiste et Gauche modérée
    if any(x in s for x in ["socialiste", "ps", "mrg", "radicaux de gauche", "majorité présidentielle", "gauche progressiste", "convention des institutions républicaines"]):
        return "Socialiste"

    # 5. Droite (RPR, Gaullistes, Indépendants, PR)
    if any(x in s for x in ["rpr", "rassemblement pour la république", "gaulliste", "cni", "indépendants et paysans", "parti républicain", "droite", "républicain indépendant", "mouvement pour la france"]):
        return "Droite"

    # 6. Centre (UDF, CDS, Radicaux valoisiens)
    if any(x in s for x in ["udf", "union pour la démocratie française", "centre", "cds", "centriste", "démocratie chrétienne", "parti radical", "radicaux-socialistes", "réformateurs"]):
        return "Centre"

    # 7. Chasse / Ruralité (Typiquement classé à droite dans les comportements électoraux)
    if any(x in s for x in ["chasse", "cpnt", "rurale"]):
        return "Droite"

    # 8. Régionalistes / Indépendantistes (Très présents dans votre liste : Corse, DOM-TOM, Bretagne...)
    if any(x in s for x in ["corse", "corsica", "kanak", "polynési", "breton", "emgann", "occitan", "catalan", "savoie", "abertzale", "guadeloup", "martiniqu", "indépendantiste", "taatiraa", "tahoeraa"]):
        return "Regionaliste"

    # 9. Sans étiquette / Indépendants purs
    if any(x in s for x in ["sans étiquette", "apolitique", "hors des partis", "indépendant", "libre", "aucun parti", "sans appartenance"]):
        return "Sans_Etiquette"

    # 10. Tout le reste (micro-partis très spécifiques non classifiables)
    return "A_EXCLURE"

# Conversion en Pandas
df = dataset.to_pandas()

# 1. Application de la normalisation
df['famille_politique'] = df['soutien'].apply(normalize_party)

# 2. Purge des données inutiles ou indéterminées
df = df[df['famille_politique'] != "A_EXCLURE"]

# 3. Homogénéisation des tailles (entre 500 et 5000 caractères)
#df = df[(df['text'].str.len() >= 500) & (df['text'].str.len() <= 5000)]

# Optionnel : Vous pouvez vérifier la répartition avant d'échantillonner
print("Répartition des forces politiques dans le corpus nettoyé :")
print(df['famille_politique'].value_counts())

# 4. Échantillonnage stratifié (3000 documents)
# Assurez-vous que les classes très minoritaires ne fassent pas planter le split
# Si une catégorie a moins de 2 représentants, la stratification renverra une erreur.
# Le filtre suivant s'assure de ne garder que les catégories ayant au moins 10 représentants.
categories_valides = df['famille_politique'].value_counts()[df['famille_politique'].value_counts() >= 10].index
df = df[df['famille_politique'].isin(categories_valides)]

df_train, _ = train_test_split(
    df, 
    train_size=3000, 
    stratify=df['famille_politique'], 
    random_state=42
)

dataset = Dataset.from_pandas(df_train)
print(f"Dataset réduit et stratifié : {len(dataset)} documents.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import unsloth
from transformers import AutoTokenizer

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def format_prompt_optimized(row):
    messages = [
        {
            "role": "system",
            "content": (
                "Tu es un expert en rhétorique politique française et un archiviste spécialisé "
                "dans l'histoire de la Ve République. Ta tâche est de rédiger une profession de "
                "foi électorale historique et convaincante. Tu dois adopter le ton, le vocabulaire "
                "et le positionnement idéologique correspondant au parti politique du candidat, "
                "en respectant scrupuleusement le contexte historique de l'année de l'élection. "
                "Le texte généré doit être structuré, persuasif, et s'adresser directement aux "
                "électeurs du département ciblé. Ne rédige que le corps du texte."
            )
        },
        {
            "role": "user",
            "content": (
                "Rédige la profession de foi à partir des caractéristiques suivantes :\n"
                f"- Candidat : {row['prenom']} {row['nom']}\n"
                f"- Profession : {row['profession']}\n"
                f"- Soutien politique : {row['soutien']}\n"
                f"- Élection : Élections législatives, Tour {row['tour']}\n"
                f"- Date : {row['date']}\n"
                f"- Département : {row['departement']}"
            )
        },
        {
            "role": "assistant",
            "content": row['text'] # Correspond à la colonne de texte nettoyé
        }
    ]
    row["formatted_text"] = tokenizer.apply_chat_template(messages, tokenize=False)
    return row

dataset = dataset.map(format_prompt_optimized)

# Analyse des longueurs
lengths = [len(tokenizer.tokenize(text)) for text in dataset['formatted_text']]
p95 = int(np.percentile(lengths, 95))
print(f"Le 95ème percentile de longueur est de {p95} tokens.")

# Ajustement automatique : on arrondit à la puissance supérieure pour l'efficacité mémoire
global_max_seq_length = 2048 if p95 > 1024 else 1024

plt.hist(lengths, bins=50, color='blue', edgecolor='black', alpha=0.7)
plt.axvline(p95, color='red', linestyle='dashed', linewidth=2, label=f'95e percentile ({p95})')
plt.title("Distribution des longueurs de séquences (tokens)")
plt.xlabel("Nombre de tokens")
plt.ylabel("Fréquence")
plt.legend()
plt.show()

# Vérification
print(dataset[0]["formatted_text"][:500])
print("...")

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 08 Apr 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

Rédige une profession de foi pour Micheline Antonucci, de profession assistance sociale, soutenu par le parti Parti socialiste unifié au tour 1 des élections législatives de 1981-06-14 dans le département : Ain.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

ELECTIONS LEGISLATIVES - 14 JUIN 1981 AIN 1e CIRCONSCRIP
...


In [8]:
# Split train / test (90% / 10%)
splits = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = splits["train"]
test_dataset  = splits["test"]

print(f"Train : {len(train_dataset)} | Test : {len(test_dataset)}")

Train : 11248 | Test : 1250


## 6. Chargement du modèle (QLoRA 4-bit)

**Points critiques pour la Tesla T4 :**
- La T4 **ne supporte pas `bf16`** (bfloat16). Il faut utiliser `fp16` (float16).
- On spécifie `torch_dtype=torch.float16` dès le chargement pour que Llama-3.2 
  charge directement ses couches non-quantifiées (lm_head, embed_tokens) en float16.
- Pas besoin de boucle de cast manuel après coup.

## 7. Entraînement

**Paramètres importants pour la T4 :**
- `fp16=True`, `bf16=False` — indispensable sur T4
- `optim="paged_adamw_32bit"` — optimiseur paginé du papier QLoRA, réduit la VRAM
- `max_seq_length=1024` — les professions de foi font souvent ~400-800 tokens,
  1024 est un bon compromis vitesse/couverture (2048 doublerait l'usage VRAM)
- `packing=True` — emballe plusieurs courtes séquences dans un même batch : accélère notablement l'entraînement

In [ ]:
import os
import time
import glob
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
from huggingface_hub import snapshot_download
from unsloth import FastLanguageModel
from transformers import EarlyStoppingCallback

# --- Paramètres dynamiques ---
OUTPUT_DIR = "./results_checkpoints"
version = time.strftime("%Y%m%d-%H%M")
REPO_ID = f"fdechamps/Llama-1B-Archelec-{version}" 

# Adaptation de la taille de lot selon le GPU détecté en cellule 1
batch_size = 8 if is_a2 else 4
accum_steps = 2 if is_a2 else 4

# --- Chargement du modèle ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=global_max_seq_length,
    dtype=None,
    load_in_4bit=False, 
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0, 
    bias="none",
    use_gradient_checkpointing="unsloth", 
    random_state=42,
)

# Nettoyage strict des datasets
colonnes_a_supprimer = [col for col in train_dataset.column_names if col != "formatted_text"]
train_dataset_clean = train_dataset.remove_columns(colonnes_a_supprimer)
test_dataset_clean = test_dataset.remove_columns(colonnes_a_supprimer)

# --- Data Collator (Masquage du prompt) ---
# Le modèle n'apprendra à prédire que ce qui suit cette balise
response_template = "<|start_header_id|>assistant<|end_header_id|>\n\n"
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size * 2,
    gradient_accumulation_steps=accum_steps,
    
    fp16=True,
    bf16=False,
    optim="adamw_8bit",
    weight_decay=0.01,
    
    dataset_text_field="formatted_text",
    max_length=global_max_seq_length,
    packing=False, # Obligatoire à False avec DataCollatorForCompletionOnlyLM
    
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    push_to_hub=True,
    hub_model_id=REPO_ID,
    hub_private_repo=True, 
    hub_strategy="checkpoint",
    report_to="none",
    # 1. EARLY STOPPING :
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset_clean,
    eval_dataset=test_dataset_clean,
    processing_class=tokenizer,
    args=training_args,
    data_collator=collator,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=5)]
)

if not os.path.exists(OUTPUT_DIR) or len(os.listdir(OUTPUT_DIR)) == 0:
    try:
        snapshot_download(repo_id=REPO_ID, local_dir=OUTPUT_DIR)
    except Exception:
        pass

checkpoints = glob.glob(f"{OUTPUT_DIR}/checkpoint-*")
if len(checkpoints) > 0:
    trainer.train(resume_from_checkpoint=True)
else:
    trainer.train()

model.save_pretrained(f"Llama-1B-Archelec-Final-{version}")
tokenizer.save_pretrained(f"Llama-1B-Archelec-Final-{version}")
trainer.push_to_hub(commit_message="Entraînement terminé")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/tmp/ipykernel_45222/1070961571.py:2: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

...

Step,Training Loss,Validation Loss
100,2.103405,2.038627
200,1.680259,1.735425
300,1.682374,1.651681
400,1.634865,1.582852
500,1.585024,1.545412
600,1.523607,1.510783
700,1.390227,1.484569
800,1.467585,1.462088
900,1.438222,1.444991
1000,1.400779,1.432052


HTTP Error 504 thrown while requesting POST https://huggingface.co/api/models/fdechamps/Llama-1B-Archelec-Checkpoints/preupload/main
[huggingface_hub.utils._http|WARNING]HTTP Error 504 thrown while requesting POST https://huggingface.co/api/models/fdechamps/Llama-1B-Archelec-Checkpoints/preupload/main
Retrying in 1s [Retry 1/5].
[huggingface_hub.utils._http|WARNING]Retrying in 1s [Retry 1/5].
/opt/python/lib/python3.13/site-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error The read operation timed out - silently ignoring the lookup for the file config.json in unsloth/Llama-3.2-1B-Instruct.
  warnings.warn(
/opt/python/lib/python3.13/site-packages/peft/utils/save_and_load.py:295: UserWarning: Could not find a config file in unsloth/Llama-3.2-1B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/fdechamps/Llama-1B-Archelec-Checkpoints/commit/d390077bb2f406efe487363e1216aca2802cbad1', commit_message='Entraînement terminé', commit_description='', oid='d390077bb2f406efe487363e1216aca2802cbad1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/fdechamps/Llama-1B-Archelec-Checkpoints', endpoint='https://huggingface.co', repo_type='model', repo_id='fdechamps/Llama-1B-Archelec-Checkpoints'), pr_revision=None, pr_num=None)

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

def test_model(prompt_text, use_lora=True):
    messages = [{"role": "user", "content": prompt_text}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    
    if use_lora:
        print("--- GÉNÉRATION (Modèle Fine-tuné) ---")
        model.generate(
            input_ids=inputs, streamer=text_streamer, max_new_tokens=512,
            temperature=0.7, top_p=0.9, repetition_penalty=1.2, do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    else:
        print("--- GÉNÉRATION (Modèle de Base Original) ---")
        with model.disable_adapter():
            model.generate(
                input_ids=inputs, streamer=text_streamer, max_new_tokens=512,
                temperature=0.7, top_p=0.9, repetition_penalty=1.2, do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
    print("\n" + "="*80 + "\n")

# 1. Comparaison sur la tâche politique
prompt_politique = (
    "Rédige la profession de foi à partir des caractéristiques suivantes :\n"
    "- Candidat : Jean Dupont\n"
    "- Profession : Chef d'entreprise\n"
    "- Soutien politique : Rassemblement pour la République\n"
    "- Élection : Élections législatives, Tour 1\n"
    "- Date : 1988\n"
    "- Département : Paris"
)

print(">>> TEST COMPARATIF : TÂCHE SPÉCIFIQUE (RHÉTORIQUE)")
test_model(prompt_politique, use_lora=True)
test_model(prompt_politique, use_lora=False)

# 2. Test des capacités intrinsèques
prompt_general = "Explique brièvement le principe de la séparation des pouvoirs."

print(">>> TEST DE CONTRÔLE : CAPACITÉS INTRINSÈQUES")
test_model(prompt_general, use_lora=False)
test_model(prompt_general, use_lora=True)

Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Génération en cours ---

ÉLECTIONS LÉGISLATIVES DU 14 JUIN 1982 - 9e CIRCONSCRIPTION DE PARIS
Jean DUPONT Chef d'Entreprise Conseiller Régional d'Ile-de-France Ancien Ministre du Général de GAULLE Vice Président de l'Ecole Nationale Supérieure Militaire et Économique Maire-Adjoint de Saint-Maur-des-Eaux Secrétaire Départemental du Parti Socialiste Membre de l'A.N.P.A.
Suppléant :
Henri LEVY Commerçant Ancien Officier Médaille d'Honneur Chevalier de l'O.C.M.S., Commandeur de la Légion d'honneur Médaillée militaire Croix des Combats et Exploits Medaillon "La Victoire" (Prusse)
Madame, Mademoiselle,
Le 10 Mai dernier, les Français ont exprimé leur volonté de changement en élisant François MITTERRAND à la tête de l'Etat.
Pour réussir cette politique nouvelle, il faut un gouvernement qui ait le courage, la détermination et la compétence nécessaire pour répondre aux aspirations des Françaises et des Fran- çais.
Dans ce but, j'ai été nommé Premier Ministre dès son premier mois de mandat,

## 8. Inférence & comparaison base vs fine-tuné

On charge les poids LoRA sur le modèle de base et on génère une profession de foi test.
On compare avec le modèle de base (sans LoRA) pour mesurer l'effet du fine-tuning.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

ADAPTER_PATH = "Llama-1B-Archelec-LoRA"

# --- Tokenizer (padding LEFT pour la génération) ---
tokenizer_inf = AutoTokenizer.from_pretrained(ADAPTER_PATH)
tokenizer_inf.padding_side = "left"

# --- Modèle de base en 4-bit ---
bnb_config_inf = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config_inf,
    torch_dtype=torch.float16,
)

# --- Chargement de l'adaptateur LoRA ---
model_ft = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

print("Modèle fine-tuné chargé.")

In [ ]:
def generate(model, tokenizer, prompt_text, max_new_tokens=400):
    """Génère une réponse à partir d'un prompt utilisateur."""
    messages = [{"role": "user", "content": prompt_text}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    # Isole la réponse de l'assistant
    return decoded.split("assistant\n")[-1].strip() if "assistant\n" in decoded else decoded


# --- Prompt de test ---
TEST_PROMPT = (
    "Rédige une profession de foi pour Jean Dupont, de profession professeur, "
    "soutenu par le parti Parti communiste français "
    "au tour 1 des élections législatives de 1981-06-14 "
    "dans le département : Paris."
)

print("=" * 60)
print("MODÈLE FINE-TUNÉ (Archelec-LoRA)")
print("=" * 60)
print(generate(model_ft, tokenizer_inf, TEST_PROMPT))

print("\n" + "=" * 60)
print("MODÈLE DE BASE (Llama-3.2-1B-Instruct sans fine-tuning)")
print("=" * 60)
with model_ft.disable_adapter():
    print(generate(model_ft, tokenizer_inf, TEST_PROMPT))